# 01 Data Download**GeoFish-AI** · runs top to bottom on a Kaggle GPU (T4 x2 or P100).**Before you run**1. *Settings → Accelerator*: **GPU T4 x2**2. *Settings → Internet*: **On** (pretrained weights are mandatory — the code raises rather than training from scratch)3. *Add Data*: attach the datasets listed in cell 3 (required ones are enforced)4. *Settings → Persistence*: **Files only**, so checkpoints survive between savesEvery stage writes into `/kaggle/working`, records itself in`/kaggle/working/outputs/logs/state.json`, and **skips itself on a re-run**.An interrupted session resumes from the last finished epoch/stage.

In [ ]:
# @title 0a · Enter the repo (clone once if this notebook was uploaded alone)import os, subprocess, sysfrom pathlib import PathREPO_DIR = Path("/kaggle/working/fishes")BRANCH = "arena/3dc076ac-fishes"def _is_repo(p): return (p / "configs" / "kaggle.yaml").exists()if _is_repo(Path.cwd()):    REPO = Path.cwd()elif _is_repo(REPO_DIR):    REPO = REPO_DIRelse:    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/Cometbuster4969/fishes.git",                    str(REPO_DIR)], check=True)    REPO = REPO_DIRos.chdir(REPO)sys.path.insert(0, str(REPO))sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,                     text=True).stdout.strip()print("repo:", REPO, "| commit:", sha)

In [ ]:
# @title 0b · Dependencies (only what the Kaggle image lacks)import osON_KAGGLE = os.path.exists("/kaggle")if ON_KAGGLE:    %pip install -q -r requirements-kaggle.txt 2>&1 | tail -5else:    print("not on Kaggle: skipped")

In [ ]:
# @title 0c · Config, seeding, resume state, mount reportfrom src.utils.notebook import bootstrapfrom src.config import describeenv = bootstrap("01_data_download", require_gpu=False)cfg, tr, log = env.cfg, env.tracker, env.logprint(describe(cfg))print()from src.utils.state import stage, validate_stateprint("stages already done:", [k for k, v in tr.state["stages"].items()                               if v.get("status") == "done"])

## 1 · Check the mountsKaggle datasets are read-only under `/kaggle/input`. The table below is generated from`configs/kaggle.yaml`; a **required** dataset that is missing raises with the exactinstructions instead of silently skipping.

In [ ]:
# @title 1a · Mount report + required-dataset enforcementimport pandas as pdfrom src.config import DatasetMissingErrorrows = []for key, d in cfg.raw["datasets"].items():    p = cfg.dataset_path(key) if not d.get("required") else None    if d.get("required"):        p = cfg.require_dataset(key)        # raises DatasetMissingError    rows.append({"dataset": key, "mounted": p is not None, "task": d["task"],                 "path": d["kaggle_input"],                 "status": "ok" if p else ("MISSING" if d.get("required") else "absent (optional)")})mounts = pd.DataFrame(rows)display(mounts)mounts.to_csv(cfg.paths.tables / "mounts.csv", index=False)

## 2 · Pretrained weights warm-upThis **proves** the network path works before any training starts: detector, classifier,embedding encoder and BioCLIP. If any checkpoint cannot be fetched, the cell raises`PretrainedDownloadError` — there is no random-init fallback anywhere in this repo.

In [ ]:
# @title 2a · Fetch one checkpoint of every family (cached in /kaggle/working/.cache)from src.utils.pretrained import (PretrainedDownloadError, timm_model,                                  ultralytics_model, open_clip_bioclip)key = "pretrained.warmup"if tr.is_done(key):    print("already verified in this working dir:", tr.get(key).metrics)else:    with stage(tr, key) as st:        if st.skip:            print("skipped (done)")        else:            report = {}            try:                m = ultralytics_model(cfg.raw["pretrained"]["detectors"]["yolo11n"])                report["detector"] = "yolo11n.pt loaded"                del m            except PretrainedDownloadError:                raise            m = timm_model(cfg.raw["pretrained"]["classifiers"]["resnet50"])            report["classifier"] = cfg.raw["pretrained"]["classifiers"]["resnet50"]            del m            bio, _, _ = open_clip_bioclip(cfg.raw["pretrained"]["bioclip"]["open_clip_tag"])            report["bioclip"] = "loaded"            del bio            print(report)            st.metric(**{k: 1 for k in report})

## 3 · Classification corpus manifestOne row per image: `path, dataset, label, dataset_label, orig_split`. The shippedtrain/val/test folders are **ignored** here — splits are rebuilt leakage-safely innotebook 02.

In [ ]:
# @title 3a · Discover every classification imagefrom src.data.discovery import discover_classificationkey = "data.classification_manifest"manifest_path = cfg.paths.data_curated / "all_classification.csv"if tr.is_done(key) and manifest_path.exists():    df_cls = pd.read_csv(manifest_path)    print("resumed:", len(df_cls), "images")else:    with stage(tr, key) as st:        if not st.skip:            df_cls = discover_classification(cfg)            manifest_path.parent.mkdir(parents=True, exist_ok=True)            df_cls.to_csv(manifest_path, index=False)            st.out(manifest_path)            st.metric(n_images=int(len(df_cls)), n_datasets=int(df_cls.dataset.nunique()),                      n_classes=int(df_cls.dataset_label.nunique()))            print(f"{len(df_cls)} images, {df_cls.dataset_label.nunique()} classes")            st.out(cfg.paths.tables / "class_counts.csv")            df_cls.groupby(["dataset", "label"]).size().rename("n_images").reset_index() \                  .to_csv(cfg.paths.tables / "class_counts.csv", index=False)display(df_cls.groupby("dataset").agg(images=("path", "size"),                                      classes=("dataset_label", "nunique")))

## 4 · Detection corpus → single class `fish`Every mounted detection dataset is converted to one class (`fish`), whatever its sourcelabel space was. Images are symlinked, labels rewritten, and the collapse is recorded sothe metrics can state exactly what the detector was trained on.

In [ ]:
# @title 4a · Build the single-class detection setimport jsonfrom src.data.discovery import build_single_class_yolo, discover_detectionkey = "data.detection_single_class"det_dir = cfg.paths.data_interim / "detection_single_class"if tr.is_done(key) and (det_dir / "data.yaml").exists():    det_stats = tr.get(key).metrics    print("resumed:", det_stats)else:    with stage(tr, key) as st:        if not st.skip:            survey = discover_detection(cfg)            det_stats = build_single_class_yolo(cfg, det_dir)            src_json = cfg.paths.tables / "detection_sources.json"            src_json.write_text(json.dumps({"survey": survey, "converted": det_stats},                                           indent=2))            st.out(det_dir / "data.yaml", src_json)            st.metric(images=det_stats["images"], boxes=det_stats["boxes"])            print(det_stats)

## 5 · Leakage-safe detection splits (grouped by near-duplicate frames)

In [ ]:
# @title 5a · Split frames 70/15/15 by pHash group, then write the YOLO data yamlfrom src.data.split import build_detection_splitskey = "data.detection_splits"if tr.is_done(key):    split_summary = tr.get(key).metrics    print("resumed:", split_summary)else:    with stage(tr, key) as st:        if not st.skip:            imgs = sorted((det_dir / "images").glob("*"))            split_summary = build_detection_splits(imgs, cfg.paths.data_splits / "detection",                                                   seed=cfg.seed)            yaml_path = det_dir / "data.yaml"            yaml_path.write_text(                f"path: {det_dir}\n"                f"train: {cfg.paths.data_splits}/detection/train_images.txt\n"                f"val: {cfg.paths.data_splits}/detection/val_images.txt\n"                f"test: {cfg.paths.data_splits}/detection/test_images.txt\n"                f"names:\n  0: fish\nnc: 1\n")            st.out(yaml_path, cfg.paths.data_splits / "detection" / "DETECTION_SPLIT_LOCK.json")            st.metric(**{k: v["n"] for k, v in split_summary["files"].items()})            print({k: v["n"] for k, v in split_summary["files"].items()})

## 6 · Integrity snapshotCounts are measured from disk (never copied from a paper or dataset card). This block alsoflags unreadable files, which would otherwise fail later in odd ways.

In [ ]:
# @title 6a · Verify decodability of a random sample + size statisticsimport randomfrom PIL import Imageimport numpy as npkey = "data.integrity"if tr.is_done(key):    print(tr.get(key).metrics)else:    with stage(tr, key) as st:        if not st.skip:            sample = random.Random(cfg.seed).sample(df_cls.path.tolist(),                                                    min(3000, len(df_cls)))            bad, sizes = [], []            for p in sample:                try:                    with Image.open(p) as im:                        sizes.append(im.size)                except Exception as exc:                    bad.append((p, str(exc)[:80]))            ws = np.array([s[0] for s in sizes]); hs = np.array([s[1] for s in sizes])            stats = {"sampled": len(sample), "unreadable": len(bad),                     "median_w": int(np.median(ws)), "median_h": int(np.median(hs)),                     "min_short_side": int(min(ws.min(), hs.min()))}            print(stats)            if bad:                print("first unreadable:", bad[:3])            st.metric(**stats)

## 7 · Provenance record + handoff

In [ ]:
# @title 7a · Write DATASETS_SNAPSHOT.json and print the run stateimport json, hashlib, datetimesnapshot = {    "generated_at": datetime.datetime.utcnow().isoformat() + "Z",    "config": str(cfg.path),    "config_sha256": hashlib.sha256(Path(cfg.path).read_bytes()).hexdigest(),    "git_commit": sha,    "mounts": mounts.to_dict(orient="records"),    "classification": {"images": int(len(df_cls)),                       "classes": int(df_cls.dataset_label.nunique()),                       "per_dataset": df_cls.dataset.value_counts().to_dict()},    "detection": det_stats,    "detection_splits": split_summary,    "pretrained_policy": cfg.raw["pretrained"]["require"],}out = cfg.paths.tables / "DATASETS_SNAPSHOT.json"out.write_text(json.dumps(snapshot, indent=2, default=str))print("wrote", out)print(tr.summary())

### Next`02_taxonomy_audit.ipynb` — taxonomy master, label map, duplicate removal, label-noisedetection, review pack. Nothing is trained before that audit passes.